In [3]:
"""
MobileViT for 11-channel archaeology patches.

Dataset structure expected on Kaggle:

FINALCNNDATA_FINAL/
├── train/
│   ├── no_site/
│   └── site/
├── val/
│   ├── no_site/
│   └── site/
├── test/
│   ├── no_site/
│   └── site/
├── metadata/
│   ├── individual/
│   └── combined/
└── _split_reports/

no_site = no-site patch
site    = site patch

Final 11-channel order used by this code:
1  Blue
2  Green
3  Red
4  NIR
5  LRM
6  Slope
7  SVF
8  Hillshade 1
9  Hillshade 2
10 Hillshade 3
11 Hillshade 4
"""

from pathlib import Path
import copy
import random

import numpy as np
import rasterio
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as T
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)


# =============================================================================
# 1. CONFIGURATION
# =============================================================================
class Config:
    DATASET_PATH = Path(
        "/kaggle/input/datasets/shreyansdeshpande/cnnfinale/FINALCNNDATA_FINAL"
    )

    NUM_CHANNELS = 11
    RAW_SIZE = 250
    MODEL_SIZE = 256

    CHANNEL_ORDER = [
        "Blue", "Green", "Red", "NIR", "LRM", "Slope",
        "SVF", "Hillshade 1", "Hillshade 2", "Hillshade 3", "Hillshade 4",
    ]

    BATCH_SIZE = 16
    NUM_EPOCHS = 30
    LEARNING_RATE = 3e-4
    WEIGHT_DECAY = 1e-4
    DROPOUT = 0.10
    NUM_WORKERS = 2
    SEED = 42

    STEM_OUT_CHANNELS = 16

    STAGE1_CFG        = (4,  32, 1)
    STAGE2_DOWN_CFG   = (4,  64, 2)
    STAGE2_REPEAT_CFG = (4,  64, 1)
    STAGE2_REPEATS    = 2

    STAGE3_DOWN_CFG = (4,  96, 2)
    STAGE4_DOWN_CFG = (4, 128, 2)
    STAGE5_DOWN_CFG = (4, 160, 2)

    MVIT3_CFG = {"in_channels": 96,  "num_blocks": 2, "projection_dim": 144, "num_heads": 2}
    MVIT4_CFG = {"in_channels": 128, "num_blocks": 4, "projection_dim": 192, "num_heads": 2}
    MVIT5_CFG = {"in_channels": 160, "num_blocks": 3, "projection_dim": 240, "num_heads": 2}

    PATCH_SIZE    = 2
    MLP_RATIO     = 2.0
    HEAD_CHANNELS = 640

    AUGMENT = True


cfg = Config()


# =============================================================================
# 2. REPRODUCIBILITY
# =============================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(cfg.SEED)


# =============================================================================
# 3. DATASET DISCOVERY
# =============================================================================
def collect_records(split_name):
    records = []
    split_path = cfg.DATASET_PATH / split_name

    if not split_path.exists():
        raise FileNotFoundError(f"Missing split folder: {split_path}")

    class_mapping = {"no_site": 0, "site": 1}

    for class_name, label in class_mapping.items():
        class_path = split_path / class_name
        if not class_path.exists():
            raise FileNotFoundError(f"Missing class folder: {class_path}")

        tif_files = sorted(
            list(class_path.glob("*.tif")) + list(class_path.glob("*.tiff"))
        )
        for file_path in tif_files:
            records.append((file_path, label))

    return records


train_records      = collect_records("train")
validation_records = collect_records("test")
test_records       = collect_records("validation")


# =============================================================================
# 4. CHANNEL STATISTICS — TRAINING SET ONLY
# =============================================================================
def calculate_channel_stats(records):
    channel_sum         = np.zeros(cfg.NUM_CHANNELS, dtype=np.float64)
    channel_squared_sum = np.zeros(cfg.NUM_CHANNELS, dtype=np.float64)
    channel_pixel_count = np.zeros(cfg.NUM_CHANNELS, dtype=np.int64)

    for file_path, _ in records:
        with rasterio.open(file_path) as raster:
            image = raster.read(masked=True).astype(np.float64)

        for channel_index in range(cfg.NUM_CHANNELS):
            valid_values = image[channel_index].compressed()
            valid_values = valid_values[np.isfinite(valid_values)]
            if valid_values.size == 0:
                continue
            channel_sum[channel_index]         += valid_values.sum()
            channel_squared_sum[channel_index] += np.square(valid_values).sum()
            channel_pixel_count[channel_index] += valid_values.size

    means     = channel_sum / channel_pixel_count
    variances = (channel_squared_sum / channel_pixel_count) - np.square(means)
    variances = np.maximum(variances, 1e-12)
    stds      = np.sqrt(variances)

    return means.astype(np.float32), stds.astype(np.float32)


channel_means, channel_stds = calculate_channel_stats(train_records)

print("Channel means (train):", channel_means)
print("Channel stds  (train):", channel_stds)


# =============================================================================
# 5. AUGMENTATION
# =============================================================================
def augment_image(image):
    """Random horizontal/vertical flip + 90-degree rotations only.
    Safe for all 11 channels including physically-absolute ones (Slope, SVF, LRM).
    """
    if np.random.rand() < 0.5:
        image = np.flip(image, axis=2).copy()
    if np.random.rand() < 0.5:
        image = np.flip(image, axis=1).copy()
    rotations = np.random.randint(0, 4)
    if rotations:
        image = np.rot90(image, k=rotations, axes=(1, 2)).copy()
    return image


# =============================================================================
# 6. PYTORCH DATASET
# =============================================================================
class ArchaeologyDataset(Dataset):
    def __init__(self, records, channel_means, channel_stds, augment=False):
        self.records = records
        self.augment = augment

        # Used for NoData / NaN pixel filling (shape: [C, 1, 1])
        # FIX: was renamed to channel_means_fill but __getitem__ still used old name
        self.channel_means_fill = channel_means.reshape(cfg.NUM_CHANNELS, 1, 1)

        # FIX: use computed training-set statistics, not hardcoded values.
        # The old code overwrote this line with a second T.Normalize using
        # undefined variables (dataset_means / dataset_stds), causing a NameError.
        self.normalizer = T.Normalize(
            mean=channel_means.tolist(),
            std=channel_stds.tolist(),
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        file_path, label = self.records[index]
        with rasterio.open(file_path) as raster:
            image = raster.read(masked=True).astype(np.float32)

        # Fill masked / NaN pixels with per-channel training mean
        image = image.filled(np.nan)
        for channel_index in range(cfg.NUM_CHANNELS):
            invalid = ~np.isfinite(image[channel_index])
            # FIX: was self.channel_means[channel_index, 0, 0] — attribute did not exist
            image[channel_index][invalid] = self.channel_means_fill[channel_index, 0, 0]

        if self.augment:
            image = augment_image(image)

        image_tensor = torch.from_numpy(image).float()
        label_tensor = torch.tensor(label, dtype=torch.float32)

        # Z-score normalise using training-set statistics
        image_tensor = self.normalizer(image_tensor)

        # Pad 250×250 → 256×256 with zeros (= channel mean in normalised space)
        if image_tensor.shape[1] == cfg.RAW_SIZE and image_tensor.shape[2] == cfg.RAW_SIZE:
            image_tensor = F.pad(image_tensor, (3, 3, 3, 3), mode="constant", value=0)

        return image_tensor, label_tensor


train_dataset      = ArchaeologyDataset(train_records,      channel_means, channel_stds, augment=cfg.AUGMENT)
validation_dataset = ArchaeologyDataset(validation_records, channel_means, channel_stds, augment=False)
test_dataset       = ArchaeologyDataset(test_records,       channel_means, channel_stds, augment=False)


# =============================================================================
# 7. DATALOADERS
# =============================================================================
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

# WeightedRandomSampler balances class frequency to ~50/50 per batch.
# FIX: pos_weight has been removed from BCEWithLogitsLoss (Section 9).
#      Using BOTH sampler AND pos_weight > 1.0 double-corrects for imbalance,
#      which biased the model to predict "site" for everything.
train_labels_list = [label for _, label in train_records]
class_counts   = np.bincount(train_labels_list)
class_weights  = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_labels_list]

sampler = WeightedRandomSampler(
    sample_weights, num_samples=len(sample_weights), replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    sampler=sampler,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=pin_memory,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=pin_memory,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=pin_memory,
)


# =============================================================================
# 8. MODEL DEFINITION
# =============================================================================
def conv_block(in_channels, out_channels, kernel_size=3, stride=2):
    return nn.Sequential(
        nn.Conv2d(
            in_channels, out_channels, kernel_size,
            stride=stride, padding=kernel_size // 2, bias=False,
        ),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
    )


class InvertedResidualBlock(nn.Module):
    def __init__(self, in_channels, expansion_ratio, out_channels, stride=1):
        super().__init__()
        expanded_channels = in_channels * expansion_ratio
        self.use_residual = (stride == 1 and in_channels == out_channels)

        self.expand = nn.Sequential(
            nn.Conv2d(in_channels, expanded_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(expanded_channels),
            nn.ReLU(),
        )
        self.depthwise = nn.Sequential(
            nn.Conv2d(
                expanded_channels, expanded_channels,
                kernel_size=3, stride=stride, padding=1,
                groups=expanded_channels, bias=False,
            ),
            nn.BatchNorm2d(expanded_channels),
            nn.ReLU(),
        )
        self.project = nn.Sequential(
            nn.Conv2d(expanded_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        m = self.expand(x)
        m = self.depthwise(m)
        m = self.project(m)
        if self.use_residual:
            m = m + x
        return m


class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn  = nn.MultiheadAttention(
            embed_dim=dim, num_heads=num_heads, dropout=dropout, batch_first=True
        )
        self.norm2  = nn.LayerNorm(dim, eps=1e-6)
        hidden_dim  = int(dim * mlp_ratio)
        self.mlp    = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x1 = self.norm1(x)
        attn_out, _ = self.attn(x1, x1, x1, need_weights=False)
        x  = x + attn_out
        x2 = self.norm2(x)
        x  = x + self.mlp(x2)
        return x


class MobileViTBlock(nn.Module):
    def __init__(self, in_channels, num_blocks, projection_dim,
                 num_heads=2, patch_size=2, mlp_ratio=2.0, dropout=0.1):
        super().__init__()
        self.patch_size  = patch_size
        self.local_rep   = nn.Sequential(
            conv_block(in_channels, in_channels, kernel_size=3, stride=1),
            nn.Conv2d(in_channels, projection_dim, kernel_size=1),
        )
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(projection_dim, num_heads, mlp_ratio, dropout)
            for _ in range(num_blocks)
        ])
        self.norm      = nn.LayerNorm(projection_dim, eps=1e-6)
        self.proj_back = conv_block(projection_dim, in_channels, kernel_size=1, stride=1)
        self.fusion    = conv_block(2 * in_channels, in_channels, kernel_size=3, stride=1)

    def forward(self, x):
        local_features = self.local_rep(x)
        B, C, H, W = local_features.shape
        p = self.patch_size

        x_unfold   = local_features.reshape(B, C, H // p, p, W // p, p)
        x_unfold   = x_unfold.permute(0, 3, 5, 2, 4, 1)
        num_patches = (H // p) * (W // p)
        x_unfold   = x_unfold.reshape(B, p * p, num_patches, C)
        x_seq      = x_unfold.reshape(B * p * p, num_patches, C)

        for block in self.transformer_blocks:
            x_seq = block(x_seq)
        x_seq = self.norm(x_seq)

        x_fold = x_seq.reshape(B, p, p, H // p, W // p, C)
        x_fold = x_fold.permute(0, 5, 3, 1, 4, 2)
        x_fold = x_fold.reshape(B, C, H, W)

        folded = self.proj_back(x_fold)
        out    = torch.cat([x, folded], dim=1)
        out    = self.fusion(out)
        return out


class MobileViT(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = conv_block(cfg.NUM_CHANNELS, cfg.STEM_OUT_CHANNELS, kernel_size=3, stride=2)

        exp, out_ch, stride = cfg.STAGE1_CFG
        self.stage1 = InvertedResidualBlock(cfg.STEM_OUT_CHANNELS, exp, out_ch, stride)
        prev_ch = out_ch

        exp, out_ch, stride = cfg.STAGE2_DOWN_CFG
        self.stage2_down = InvertedResidualBlock(prev_ch, exp, out_ch, stride)
        prev_ch = out_ch

        exp, rep_ch, rep_stride = cfg.STAGE2_REPEAT_CFG
        self.stage2 = nn.Sequential(*[
            InvertedResidualBlock(prev_ch, exp, rep_ch, rep_stride)
            for _ in range(cfg.STAGE2_REPEATS)
        ])
        prev_ch = rep_ch

        exp, out_ch, stride = cfg.STAGE3_DOWN_CFG
        self.stage3_down = InvertedResidualBlock(prev_ch, exp, out_ch, stride)
        self.stage3_mvit = MobileViTBlock(
            patch_size=cfg.PATCH_SIZE, mlp_ratio=cfg.MLP_RATIO,
            dropout=cfg.DROPOUT, **cfg.MVIT3_CFG,
        )
        prev_ch = out_ch

        exp, out_ch, stride = cfg.STAGE4_DOWN_CFG
        self.stage4_down = InvertedResidualBlock(prev_ch, exp, out_ch, stride)
        self.stage4_mvit = MobileViTBlock(
            patch_size=cfg.PATCH_SIZE, mlp_ratio=cfg.MLP_RATIO,
            dropout=cfg.DROPOUT, **cfg.MVIT4_CFG,
        )
        prev_ch = out_ch

        exp, out_ch, stride = cfg.STAGE5_DOWN_CFG
        self.stage5_down = InvertedResidualBlock(prev_ch, exp, out_ch, stride)
        self.stage5_mvit = MobileViTBlock(
            patch_size=cfg.PATCH_SIZE, mlp_ratio=cfg.MLP_RATIO,
            dropout=cfg.DROPOUT, **cfg.MVIT5_CFG,
        )
        prev_ch = out_ch

        self.conv_head   = conv_block(prev_ch, cfg.HEAD_CHANNELS, kernel_size=1, stride=1)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        # FIX: dropout added before classifier for regularisation
        self.dropout     = nn.Dropout(cfg.DROPOUT)
        self.classifier  = nn.Linear(cfg.HEAD_CHANNELS, 1)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2_down(x)
        x = self.stage2(x)
        x = self.stage3_down(x)
        x = self.stage3_mvit(x)
        x = self.stage4_down(x)
        x = self.stage4_mvit(x)
        x = self.stage5_down(x)
        x = self.stage5_mvit(x)
        x = self.conv_head(x)
        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        return self.classifier(x)


# =============================================================================
# 9. MODEL / LOSS / OPTIMIZER
# =============================================================================
model = MobileViT().to(device)

# FIX: pos_weight removed entirely.
#      WeightedRandomSampler already equalises class frequency to ~50/50.
#      Adding pos_weight=1.3 on top of that gave an effective weight of ~2.4,
#      which collapsed the model into predicting "site" for everything.
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.AdamW(
    model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY
)

# FIX: factor 0.1 → 0.5  (10× reduction was too aggressive, killed LR in 2 triggers)
#      patience 3 → 4
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=4)


# =============================================================================
# 10. TRAINING / VALIDATION LOOP
# =============================================================================
def run_epoch(model, loader, training):
    model.train() if training else model.eval()

    total_loss    = 0.0
    total_samples = 0
    all_labels    = []
    all_preds     = []

    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).view(-1, 1)

            if training:
                optimizer.zero_grad(set_to_none=True)

            outputs = model(images)
            loss    = criterion(outputs, labels)

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss    += loss.item() * images.size(0)
            total_samples += images.size(0)

            preds = (torch.sigmoid(outputs) >= 0.67).long()
            all_labels.extend(labels.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy().flatten())

    avg_loss   = total_loss / total_samples
    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)

    accuracy = accuracy_score(all_labels, all_preds)

    # FIX: per-class precision / recall / F1 for BOTH classes.
    #      Old code only reported site-class metrics (default pos_label=1),
    #      making it impossible to detect whether no_site recall was recovering.
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, labels=[0, 1], zero_division=0
    )
    # prec[0], rec[0], f1[0]  →  no_site class
    # prec[1], rec[1], f1[1]  →  site class

    conf_mat = confusion_matrix(all_labels, all_preds, labels=[0, 1])

    return avg_loss, accuracy, prec, rec, f1, conf_mat


# =============================================================================
# 11. TRAINING LOOP
# =============================================================================
# FIX: best model now tracked by macro-F1 (average of both classes),
#      not val_loss. While collapsed, val_loss can decrease even when the
#      model predicts everything as "site".
best_macro_f1    = -1.0
best_model_state = copy.deepcopy(model.state_dict())

print("\n" + "=" * 70)
print("TRAINING")
print(f"Device: {device} | Sampler: balanced | pos_weight: none")
print("=" * 70)

for epoch in range(cfg.NUM_EPOCHS):
    train_loss, train_acc, train_prec, train_rec, train_f1, train_conf = \
        run_epoch(model, train_loader, training=True)

    val_loss, val_acc, val_prec, val_rec, val_f1, val_conf = \
        run_epoch(model, validation_loader, training=False)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    val_macro_f1 = val_f1.mean()
    if val_macro_f1 > best_macro_f1:
        best_macro_f1    = val_macro_f1
        best_model_state = copy.deepcopy(model.state_dict())

    print(f"\n--- Epoch {epoch + 1:02d}/{cfg.NUM_EPOCHS} | LR: {current_lr:.2e} ---")
    print(
        f"TRAIN | Loss: {train_loss:.4f} | Acc: {train_acc*100:.1f}%\n"
        f"       no_site → P:{train_prec[0]:.3f} R:{train_rec[0]:.3f} F1:{train_f1[0]:.3f}\n"
        f"       site    → P:{train_prec[1]:.3f} R:{train_rec[1]:.3f} F1:{train_f1[1]:.3f}\n"
        f"       Conf Mat:\n{train_conf}"
    )
    print(
        f"VAL   | Loss: {val_loss:.4f} | Acc: {val_acc*100:.1f}% | Macro-F1: {val_macro_f1:.3f}\n"
        f"       no_site → P:{val_prec[0]:.3f} R:{val_rec[0]:.3f} F1:{val_f1[0]:.3f}  ← watch this\n"
        f"       site    → P:{val_prec[1]:.3f} R:{val_rec[1]:.3f} F1:{val_f1[1]:.3f}\n"
        f"       Conf Mat:\n{val_conf}"
    )

model.load_state_dict(best_model_state)
print(f"\nBest val macro-F1: {best_macro_f1:.4f}")


# =============================================================================
# 12. SAVE MODEL
# =============================================================================
model_save_path = "/kaggle/working/mobilevit_11channel_best.pth"
torch.save({"model_state_dict": model.state_dict()}, model_save_path)
print("Saved model:", model_save_path)

Channel means (train): [0.03558573 0.04399084 0.03591461 0.25498414 0.01549541 7.4376154
 0.9299269  0.6347714  0.6314146  0.63250005 0.6360352 ]
Channel stds  (train): [0.00893266 0.01691685 0.01417639 0.09966885 0.72485    6.348506
 0.05267664 0.08987585 0.09190632 0.09012205 0.09248438]

TRAINING
Device: cuda | Sampler: balanced | pos_weight: none

--- Epoch 01/30 | LR: 3.00e-04 ---
TRAIN | Loss: 0.6951 | Acc: 50.3%
       no_site → P:0.499 R:0.894 F1:0.640
       site    → P:0.533 R:0.119 F1:0.194
       Conf Mat:
[[1128  134]
 [1134  153]]
VAL   | Loss: 0.7226 | Acc: 76.7% | Macro-F1: 0.503
       no_site → P:0.795 R:0.949 F1:0.865  ← watch this
       site    → P:0.323 R:0.090 F1:0.141
       Conf Mat:
[[391  21]
 [101  10]]

--- Epoch 02/30 | LR: 3.00e-04 ---
TRAIN | Loss: 0.6934 | Acc: 52.1%
       no_site → P:0.510 R:0.924 F1:0.657
       site    → P:0.626 R:0.125 F1:0.209
       Conf Mat:
[[1168   96]
 [1124  161]]
VAL   | Loss: 0.7529 | Acc: 70.2% | Macro-F1: 0.535
       no

In [4]:
import os

statsPath = "/kaggle/working/channel_stats.npz"

if os.path.exists(statsPath):
    os.remove(statsPath)
    print("Deleted stale channel_stats.npz")

Deleted stale channel_stats.npz


In [16]:
import os

print("Contents of /kaggle/input:")
print(os.listdir("/kaggle/input/datasets/shreyansdeshpande/cnnfinale/FINALCNNDATA_FINAL"))

# Verify full path dynamically
input_base = "/kaggle/input"
for item in os.listdir(input_base):
    if "cnnfinale" in item.lower():
        full_path = os.path.join(input_base, item, "FINALCNNDATA_FINAL")
        print(f"Found dataset at: {full_path}")
        print("Subfolders:", os.listdir(full_path))

Contents of /kaggle/input:
['_split_reports', 'validation', 'test', 'train']
